# 3. Modelagem Não Supervisionada — Clusterização e PCA

## Objetivo
Aplicar técnicas não supervisionadas para descobrir padrões estruturais nos dados de voos:

- **KMeans Clustering**: Agrupar aeroportos por perfil operacional (volume, atrasos, distância)
- **PCA (Análise de Componentes Principais)**: Reduzir dimensionalidade e identificar as variáveis que mais contribuem para a variância dos dados

### Por que não supervisionado?
Diferente dos modelos supervisionados que preveem um alvo definido, aqui buscamos **padrões ocultos** que podem revelar clusters naturais de comportamento e reduzir a complexidade dimensional.

## 3.1 Setup e Carregamento dos Dados

In [7]:
import sys
sys.path.insert(0, "..")

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_clean_data
from src.feature_engineering import prepare_features
from src.unsupervised import run_clustering, run_pca_analysis, create_airport_profile

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

# Carregar dados
flights, airlines, airports = load_clean_data()
flights = prepare_features(flights)
print(f"Dataset: {flights.shape[0]:,} voos × {flights.shape[1]} colunas")

Dataset: 5,714,008 voos × 51 colunas


## 3.2 Clusterização de Aeroportos (KMeans)

Agrupamos aeroportos de origem pelo seu **perfil operacional**:
- Volume total de voos
- Atraso médio na chegada
- Percentual de voos atrasados
- Distância média dos voos
- Tempo de voo médio

**Método**: KMeans com seleção automática do número de clusters (k) via:
- **Método do Cotovelo (Elbow)**: Ponto de inflexão na inércia
- **Silhouette Score**: Coesão e separação dos clusters (quanto maior, melhor)

> Apenas aeroportos com ≥ 1.000 voos são considerados para garantir representatividade estatística.

In [8]:
# Executar clusterização
airport_profiles, cluster_info = run_clustering(flights)
print(f"\nNúmero ótimo de clusters: {cluster_info['best_k']}")
print(f"Aeroportos analisados: {len(airport_profiles)}")

  >> Salvo: outputs/figures/elbow_silhouette.png

Melhor k (silhouette): 2

=== Perfil Médio por Cluster ===
         total_voos  media_atraso_partida  media_atraso_chegada  pct_atrasados  media_distancia  media_taxi_out  num_companhias  num_destinos
CLUSTER                                                                                                                                      
0          62422.96                  8.11                  2.43           0.17           938.44           16.75           10.18         72.57
1           6170.21                  6.71                  3.49           0.16           542.46           13.37            4.68         12.07
  >> Salvo: outputs/figures/clustering_results.png

--- DBSCAN (comparação) ---
Clusters DBSCAN: 5
Outliers (ruído): 28 aeroportos
Silhouette (DBSCAN, excluindo ruído): 0.186

Número ótimo de clusters: 2
Aeroportos analisados: 297


### Perfil dos clusters

Analisando as características médias de cada cluster para interpretar os agrupamentos.

In [9]:
# Perfil medio por cluster (KMeans)
profile_cols = [c for c in airport_profiles.columns if c not in ["ORIGIN_AIRPORT", "CLUSTER", "DBSCAN_LABEL"]]
cluster_profile = airport_profiles.groupby("CLUSTER")[profile_cols].mean().round(2)
print("Perfil medio de cada cluster (KMeans):\n")
display(cluster_profile)

# Contagem de aeroportos por cluster
print(f"\nAeroportos por cluster (KMeans):")
print(airport_profiles["CLUSTER"].value_counts().sort_index())

# Resultado DBSCAN
print(f"\n--- Comparacao com DBSCAN ---")
print(f"Clusters DBSCAN: {cluster_info['n_clusters_dbscan']}")
n_noise = (airport_profiles["DBSCAN_LABEL"] == -1).sum()
print(f"Outliers (ruido DBSCAN): {n_noise} aeroportos")
if n_noise < len(airport_profiles):
    print("\nAeroportos identificados como outliers pelo DBSCAN:")
    outlier_airports = airport_profiles[airport_profiles["DBSCAN_LABEL"] == -1]["ORIGIN_AIRPORT"].tolist()
    print(f"  {outlier_airports}")

Perfil medio de cada cluster (KMeans):



,total_voos,media_atraso_partida,media_atraso_chegada,mediana_atraso_chegada,std_atraso_chegada,pct_atrasados,media_distancia,media_taxi_out,media_taxi_in,num_companhias,num_destinos
CLUSTER,,,,,,,,,,,
0,62422.96,8.11,2.43,-5.87,36.45,0.17,938.44,16.75,7.39,10.18,72.57
1,6170.21,6.71,3.49,-6.02,41.58,0.16,542.46,13.37,8.91,4.68,12.07



Aeroportos por cluster (KMeans):
CLUSTER
0     67
1    230
Name: count, dtype: int64

--- Comparacao com DBSCAN ---
Clusters DBSCAN: 5
Outliers (ruido DBSCAN): 28 aeroportos

Aeroportos identificados como outliers pelo DBSCAN:
  ['10397', '11057', '12173', '12478', '12953', '13232', '13303', 'ASE', 'ATL', 'DEN', 'DFW', 'DTW', 'EGE', 'EWR', 'HPN', 'IAD', 'IAH', 'ISN', 'ITO', 'JFK', 'LGA', 'MIA', 'MSP', 'ORD', 'PBI', 'PDX', 'SEA', 'SLC']


## 3.3 Análise de Componentes Principais (PCA)

**Objetivo**: Reduzir a dimensionalidade dos dados numéricos e identificar quais variáveis explicam a maior parte da variância.

**O que o PCA revela**:
- **Variância explicada**: Quantos componentes são necessários para explicar 90% da variância
- **Loadings**: Peso de cada variável original em cada componente principal
- **Interpretação**: Quais dimensões latentes existem nos dados (ex: dimensão de "operação do voo", "atrasos", "distância")

> Os dados são padronizados (StandardScaler) antes do PCA para evitar viés de escala.

In [10]:
# Executar PCA
pca_results = run_pca_analysis(flights)
print(f"\nComponentes para 90% de variância: {pca_results['n_components_90']}")


Componentes para 90% da variância: 7
  >> Salvo: outputs/figures/pca_analysis.png

Componentes para 90% de variância: 7


### Loadings — Contribuição das variáveis

A tabela de loadings mostra o peso de cada variável original nos primeiros componentes principais. Valores altos (positivos ou negativos) indicam forte contribuição.

In [11]:
# Exibir tabela de loadings
loadings = pca_results["loadings"]
print(f"Loadings dos {loadings.shape[1]} primeiros componentes:\n")
display(loadings.style.background_gradient(cmap="RdBu_r", vmin=-1, vmax=1))

Loadings dos 2 primeiros componentes:



,PC1,PC2
DEPARTURE_DELAY,0.055835,0.548542
ARRIVAL_DELAY,0.046033,0.563401
DISTANCE,0.490254,-0.064257
SCHEDULED_TIME,0.493830,-0.061244
ELAPSED_TIME,0.497524,-0.036160
AIR_TIME,0.494208,-0.057404
TAXI_OUT,0.092108,0.128871
TAXI_IN,0.072675,0.054839
AIR_SYSTEM_DELAY,0.055854,0.244546
SECURITY_DELAY,0.005219,0.019080


## 3.4 Deteccao de Anomalias (Isolation Forest)

**Objetivo**: Identificar voos com comportamento anomalo usando Isolation Forest, que isola observacoes que diferem significativamente do padrao normal.

**Abordagem**:
- Variaveis usadas: DEPARTURE_DELAY, ARRIVAL_DELAY, DISTANCE, SCHEDULED_TIME, TAXI_OUT, TAXI_IN
- Contaminacao estimada: 5% (proporcao de anomalias esperada)
- Amostra de 200K voos para viabilidade computacional

**O que esperamos**: Voos com atrasos extremos, rotas atipicas ou tempos operacionais fora do normal.

In [12]:
from src.unsupervised import run_anomaly_detection

anomaly_results = run_anomaly_detection(flights)

print(f"\nAnomalias detectadas: {anomaly_results['n_anomalies']:,} ({anomaly_results['pct_anomalies']:.1f}%)")
print(f"\nPerfil comparativo (anomalias vs normais):")
display(anomaly_results["profile"])


Detecção de Anomalias (Isolation Forest):
  Normais: 190,000 (95.0%)
  Anomalias: 10,000 (5.0%)

Perfil médio:
          DEPARTURE_DELAY  ARRIVAL_DELAY  DISTANCE  SCHEDULED_TIME  TAXI_OUT  TAXI_IN
ANOMALY                                                                              
Anomalia            94.90         102.38   1380.18          216.30     27.58    13.29
Normal               4.66          -0.88    795.60          138.04     15.46     7.12
  >> Salvo: outputs/figures/anomaly_detection.png

Anomalias detectadas: 10,000 (5.0%)

Perfil comparativo (anomalias vs normais):


,DEPARTURE_DELAY,ARRIVAL_DELAY,DISTANCE,SCHEDULED_TIME,TAXI_OUT,TAXI_IN
ANOMALY,,,,,,
Anomalia,94.90,102.38,1380.18,216.30,27.58,13.29
Normal,4.66,-0.88,795.60,138.04,15.46,7.12


## 3.5 Analise Critica — Modelagem Nao Supervisionada

### Interpretacao dos Resultados

**Clusterizacao (KMeans)**:
- Metodo do cotovelo e silhouette score determinaram o k otimo
- Clusters revelam perfis distintos: hubs de alto volume com atrasos moderados vs aeroportos regionais
- Visualizacao PCA 2D confirma separabilidade dos grupos
- Heatmap de perfis medios permite interpretacao qualitativa de cada cluster

**DBSCAN (comparativo)**:
- Detecta automaticamente o numero de clusters e identifica outliers (ruido)
- Aeroportos marcados como ruido pelo DBSCAN sao tipicamente extremos operacionais (volumetria muito alta ou muito baixa)
- Complementa o KMeans por nao assumir clusters esfericos

**PCA**:
- Variancia acumulada indica quantos componentes sao necessarios para reter 90% da informacao
- Primeiras componentes capturam: (1) dimensao de tempo de voo/distancia e (2) dimensao de atrasos
- Reducao dimensional pode ser usada como pre-processamento para outros modelos

**Deteccao de Anomalias (Isolation Forest)**:
- Identificou ~5% dos voos como anomalos — tipicamente voos com atrasos extremos, tempos de taxi anormais ou distancias atipicas
- O perfil medio das anomalias mostra valores significativamente maiores de atraso em comparacao aos voos normais
- Visualizacao em espaco PCA 2D confirma que anomalias estao em regioes perifericas
- Pode ser usado como filtro de qualidade de dados ou sistema de alerta

### Limitacoes
- KMeans assume clusters esfericos e tamanhos similares — DBSCAN mitiga parcialmente isso
- Apenas variaveis numericas sao usadas no PCA e na deteccao de anomalias
- Amostragem pode nao captar subclusters de aeroportos minoritarios
- Isolation Forest e sensivel ao parametro de contaminacao (5% e uma estimativa)
- Nao ha rotulo verdadeiro para anomalias — avaliacao e qualitativa